# Laboratorio visual del ETL — Business Intelligence & Analytics

**Objetivo:** entender, paso a paso, qué hace el módulo `src/etl/procesar_datos.py` del TP: fuentes Excel → limpieza → unión → notas → alertas → tablas analíticas.

**No copiamos ni reescribimos la lógica:** importamos las funciones reales del proyecto y mostramos resultados intermedios. Si cambia el script, este laboratorio utiliza su implementación actual al reiniciar el entorno y volver a clonar.

**Orden:** ejecutar las celdas de arriba hacia abajo. La rama elegida es `matias`; no modifica `main`. Este notebook lee los Excel originales, los cuales **no están guardados en GitHub**. Las transformaciones intermedias se mantienen en memoria y al final se puede ejecutar el ETL completo.

> Las cifras y gráficas dependen de los archivos que cargues. Los ejemplos de vista previa muestran un máximo de algunas filas para no saturar Colab.

## 0. Preparar Colab y clonar el repositorio privado

**Antes de ejecutar:** en Colab abrí el panel de **Secrets** (ícono de llave), creá `GITHUB_TOKEN` con un token personal de GitHub de acceso **solo lectura** a `bi-analytics-tp`, y permití el acceso del notebook al secret. No escribas ni publiques el token en una celda.

El código lo pasa como autenticación temporal a Git. No queda escrito en la URL remota, en el notebook ni en el resultado de las celdas. Si ya tenés el repositorio clonado en `/content/bi-analytics-tp`, lo reutiliza. El token únicamente es necesario para clonar el repositorio privado.

In [ ]:
from pathlib import Path
import os, sys, subprocess, base64

REPO = "https://github.com/matiasjuarez95/bi-analytics-tp.git"
RAMA = "matias"
PROJECT_DIR = Path("/content/bi-analytics-tp")

if not PROJECT_DIR.is_dir():
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception as exc:
        raise RuntimeError("Agregá GITHUB_TOKEN en Secrets de Colab y habilitá el acceso del notebook.") from exc
    if not token:
        raise RuntimeError("Falta el secret GITHUB_TOKEN en Colab.")
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = os.environ.copy()
    env["GIT_CONFIG_COUNT"] = "1"
    env["GIT_CONFIG_KEY_0"] = "http.extraheader"
    env["GIT_CONFIG_VALUE_0"] = f"AUTHORIZATION: basic {auth}"
    subprocess.run(["git", "clone", "--quiet", "--branch", RAMA, REPO, str(PROJECT_DIR)],
                   check=True, env=env)
    del token, auth, env
else:
    print("Se reutiliza el repositorio existente; reiniciá el entorno para clonar una versión nueva.")

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-r", "requirements.txt"], check=True)
from src.etl import procesar_datos as etl
import pandas as pd
from IPython.display import display
print("Repositorio:", PROJECT_DIR)
print("Rama:", subprocess.check_output(["git", "branch", "--show-current"], text=True).strip())

## 1. Leer los Excel directamente desde Google Drive

Los datos de entrada están en esta [carpeta compartida](https://drive.google.com/drive/folders/1XUbw9IHaKuRVce5qrULsp31V2HRnQI9W).

Esta celda abre la autenticación de Google, accede **por el ID de la carpeta** (incluso si está en «Compartidos conmigo») y copia los archivos `.xlsx` al almacenamiento temporal de Colab, conservando las subcarpetas. Así, el ETL puede abrirlos con `pandas.read_excel`. No hace falta descargar ni subir ZIP, ni crear un acceso directo en «Mi unidad».

**Requisito:** autenticarse con una cuenta de Google que tenga permiso de lectura sobre la carpeta. Estos archivos se descargan solo a la sesión temporal de Colab; no se incorporan al repositorio de GitHub. Al reiniciar el entorno, habrá que volver a ejecutar esta celda.

In [ ]:
from pathlib import Path
from io import BytesIO
from google.colab import auth
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import re

# Es el ID de la carpeta de Drive, no una ruta del sistema de archivos.
DRIVE_FOLDER_ID = "1XUbw9IHaKuRVce5qrULsp31V2HRnQI9W"
RAW_DIR = Path("/content/datos_lms")
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Autenticación de solo lectura con la cuenta Google que tiene acceso a la carpeta.
auth.authenticate_user()
credenciales, _ = google.auth.default(
    scopes=["https://www.googleapis.com/auth/drive.readonly"]
)
drive = build("drive", "v3", credentials=credenciales, cache_discovery=False)

def listar_hijos(folder_id):
    archivos = []
    pagina = None
    while True:
        respuesta = drive.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields="nextPageToken,files(id,name,mimeType)",
            pageSize=1000,
            pageToken=pagina,
            supportsAllDrives=True,
            includeItemsFromAllDrives=True
        ).execute()
        archivos.extend(respuesta.get("files", []))
        pagina = respuesta.get("nextPageToken")
        if not pagina:
            break
    return sorted(archivos, key=lambda x: x["name"])

def descargar_carpeta(folder_id, destino):
    cantidad = 0
    for elemento in listar_hijos(folder_id):
        nombre = elemento["name"]
        # No permitimos rutas absolutas ni componentes de directorio en los nombres remotos.
        if nombre in (".", "..") or "/" in nombre or chr(92) in nombre:
            raise ValueError(f"Nombre inseguro en Drive: {nombre}")
        ruta = destino / nombre
        if elemento["mimeType"] == "application/vnd.google-apps.folder":
            ruta.mkdir(parents=True, exist_ok=True)
            cantidad += descargar_carpeta(elemento["id"], ruta)
        elif nombre.lower().endswith(".xlsx"):
            if not ruta.exists():
                solicitud = drive.files().get_media(fileId=elemento["id"], supportsAllDrives=True)
                with ruta.open("wb") as salida:
                    downloader = MediaIoBaseDownload(salida, solicitud)
                    terminado = False
                    while not terminado:
                        _, terminado = downloader.next_chunk()
            cantidad += 1
    return cantidad

cantidad = descargar_carpeta(DRIVE_FOLDER_ID, RAW_DIR)
print(f"Excel disponibles: {cantidad}")
print("Carpeta local para el ETL:", RAW_DIR)

# Comprueba que la carpeta incluya los reportes obligatorios de las 3 materias.
archivos = etl.encontrar_archivos(str(RAW_DIR))
print("Asignaturas detectadas:", ", ".join(archivos.keys()))

## 2. Inventario: ¿qué archivos usa cada asignatura?

`encontrar_archivos()` reconoce nombres con y sin tildes, verifica los tres tipos de reporte y la matrícula, y detiene el proceso si encuentra una fuente obligatoria ausente o ambigua.

In [ ]:
etl.reiniciar_registro()
archivos = etl.encontrar_archivos(str(RAW_DIR))
inventario = []
for materia, fuentes in archivos.items():
    inventario.append({
        "asignatura": materia,
        "matricula": Path(fuentes["matricula"]).name,
        "participacion": Path(fuentes["participacion"]).name,
        "trazabilidad": Path(fuentes["trazabilidad"]).name,
        "archivos_calificaciones": len(fuentes["calificaciones"]),
    })
display(pd.DataFrame(inventario))
print("Total de Excel detectados:", sum(3 + len(v["calificaciones"]) for v in archivos.values()))

## 3. Matrícula: altas, bajas y re-inscripciones

Una persona puede tener varios movimientos. `leer_matricula()` interpreta el archivo; `resolver_matricula()` conserva **una fila vigente por estudiante y materia**, y mantiene todos los movimientos para auditoría.

**Pregunta para explorar:** ¿qué registros quedaron en Baja y cuáles en Aceptada?

In [ ]:
mats, movimientos = [], []
for asig in etl.ASIGNATURAS:
    vig, mov = etl.resolver_matricula(etl.leer_matricula(archivos[asig]["matricula"], asig), asig)
    mats.append(vig)
    movimientos.append(mov)
matricula = pd.concat(mats, ignore_index=True)
matricula_movimientos = pd.concat(movimientos, ignore_index=True)
display(matricula.head(8))
display(pd.crosstab(matricula["asignatura"], matricula["estado_insc"]))
print("Personas con varios movimientos:", matricula_movimientos.loc[
    matricula_movimientos["n_movimientos"] > 1, ["asignatura","id_estudiante"]
].drop_duplicates().shape[0])
display(matricula_movimientos.loc[matricula_movimientos["n_movimientos"] > 1,
    ["asignatura","id_estudiante","estado_insc","fecha_inscripcion","es_vigente"]].head(12))

## 4. Participación: ¿quién ingresó y cuánto participó?

Las fechas, los tiempos de uso y las interacciones se normalizan. Se separan los perfiles de alumnos del personal y se detectan registros con cero interacción.

In [ ]:
participaciones, personal_participacion = [], []
for asig in etl.ASIGNATURAS:
    df = etl.leer_participacion(archivos[asig]["participacion"], asig)
    alumnos, personal = etl.procesar_participacion(df, asig)
    participaciones.append(alumnos)
    personal_participacion.append(personal)
participacion = pd.concat(participaciones, ignore_index=True)
display(participacion[["asignatura","id_estudiante","interacciones",
                        "tiempo_uso_min","nunca_ingreso"]].head(10))
display(participacion.groupby("asignatura", as_index=False).agg(
    estudiantes=("id_estudiante","nunique"),
    interacciones=("interacciones","sum"),
    nunca_ingresaron=("nunca_ingreso","sum"),
    tiempo_medio_min=("tiempo_uso_min","mean")
).round(1))

## 5. Trazabilidad: eventos, duplicados y tipos de interacción

A diferencia de la participación, la trazabilidad contiene **muchos eventos por estudiante**. El ETL conserva eventos idénticos y los marca como duplicados *potenciales*; no asume que sean errores.

In [ ]:
trazas, personal_trazabilidad = [], []
for asig in etl.ASIGNATURAS:
    tr, n_original = etl.leer_trazabilidad(archivos[asig]["trazabilidad"], asig)
    alumnos, personal = etl.separar_alumnos_personal(tr, "trazabilidad", asig)
    trazas.append(alumnos)
    personal_trazabilidad.append(personal)
trazabilidad = pd.concat(trazas, ignore_index=True)
display(trazabilidad[["asignatura","id_estudiante","fecha","accion",
                       "tipo_interaccion","duplicado_potencial"]].head(10))
display(trazabilidad.groupby(["asignatura","tipo_interaccion"]).size().unstack(fill_value=0))
print("Eventos potencialmente repetidos:", int(trazabilidad["duplicado_potencial"].sum()))

In [ ]:
import plotly.express as px
tipo_por_materia = (trazabilidad.groupby(["asignatura","tipo_interaccion"])
                    .size().reset_index(name="eventos"))
fig = px.bar(tipo_por_materia, x="asignatura", y="eventos",
             color="tipo_interaccion", barmode="stack",
             title="Eventos del LMS por tipo y asignatura")
fig.show()

## 6. Calificaciones: formato largo y recuperatorios

Cada archivo Excel representa un instrumento. El ETL lee los archivos, extrae nota y máximo, y genera **una fila por estudiante e instrumento**.

Los recuperatorios tienen una lógica particular: el código señala cuáles notas se usan y cuáles no, pero no elimina los registros históricos.

In [ ]:
calificaciones_list = []
for asig in etl.ASIGNATURAS:
    for ruta in archivos[asig]["calificaciones"]:
        calificaciones_list.append(etl.leer_calificaciones(ruta, asig))
calificaciones_crudas = pd.concat(calificaciones_list, ignore_index=True)
calificaciones = etl.aplicar_recuperatorios(calificaciones_crudas, modo="docente")
display(calificaciones[["id_estudiante","asignatura","instrumento","puntos","maximo",
                         "nota_100","es_recuperatorio","usa_en_nota_final"]].head(12))
display(calificaciones.groupby(["asignatura","tipo_instrumento"]).agg(
    registros=("id_estudiante","size"),
    notas_utilizadas=("usa_en_nota_final","sum")
).reset_index())
display(calificaciones.loc[calificaciones["es_recuperatorio"],
     ["id_estudiante","instrumento","nota_100","usa_en_nota_final"]].head(15))

## 7. Universo de alumnos: integración sin duplicar estudiantes

El universo es la unión de quienes aparecen en matrícula, participación, trazabilidad o calificaciones, usando `(id_estudiante, asignatura)` como clave.

Un alumno puede estar en varias materias: lo contamos una vez **por materia**, no una vez por evento.

In [ ]:
universo = pd.concat([
    matricula[["id_estudiante","asignatura"]],
    participacion[["id_estudiante","asignatura"]],
    calificaciones[["id_estudiante","asignatura"]],
    trazabilidad[["id_estudiante","asignatura"]],
], ignore_index=True).drop_duplicates().reset_index(drop=True)
print("Filas del universo:", len(universo))
print("Claves duplicadas:", universo.duplicated(["id_estudiante","asignatura"]).sum())
display(universo.groupby("asignatura").size().rename("alumnos").reset_index())

## 8. Notas finales: real vs. estimado

**Álgebra** acumula puntos reales, pero también calcula una proyección sobre los instrumentos disponibles. Las proyecciones **no determinan el estado definitivo**.

**Probabilidad y Algoritmos** usan promedios; el ETL distingue promedio observado y promedio considerando cero para exámenes presentes que no se rindieron.

El `estado_provisorio` y el `estado_definitivo` no significan lo mismo.

In [ ]:
notas_finales = etl.calcular_notas_finales(calificaciones, universo, cursada_cerrada=False)
columnas = ["asignatura","id_estudiante","instrumentos_rendidos","nota_a_la_fecha",
            "estado_provisorio","estado_definitivo"]
display(notas_finales[columnas].head(12))
algebra = notas_finales[notas_finales["asignatura"] == "Álgebra"]
display(algebra[["id_estudiante","puntos_acumulados","puntos_maximos_observados",
                 "proyeccion_120_observados","estado_provisorio","estado_definitivo"]].head(12))
display(pd.crosstab(notas_finales["asignatura"], notas_finales["estado_provisorio"]))

## 9. Tabla integrada: situación de cada estudiante

El ETL agrega por estudiante-materia los eventos, días activos, primer y último evento; luego combina todo con matrícula, participación y notas.

**Regla clave:** `situacion_estudiante` debe tener exactamente una fila por `(id_estudiante, asignatura)`.

In [ ]:
eventos = trazabilidad.groupby(["id_estudiante","asignatura"]).agg(
    eventos_trazabilidad=("fecha","size"),
    eventos_repetidos_potenciales=("duplicado_potencial","sum"),
    dias_activos=("fecha_dia","nunique"),
    primer_evento=("fecha","min"),
    ultimo_evento=("fecha","max"),
).reset_index()
situacion = etl.construir_situacion(universo, matricula, participacion, eventos, notas_finales)
print("Filas del universo:",len(universo),"| Filas del modelo:",len(situacion))
assert not situacion.duplicated(["id_estudiante","asignatura"]).any()
display(situacion[["asignatura","id_estudiante","matriculado",
                   "tiene_actividad","situacion_matricula","estado_provisorio","semaforo"]].head(15))
display(pd.crosstab(situacion["asignatura"], situacion["situacion_matricula"]))

## 10. Calidad: ¿qué se descartó y qué quedó marcado?

El registro de calidad `LOG` guarda decisiones, afectados y balances. `DESCARTADOS` conserva filas descartadas. Mirá las causas antes de confiar en una métrica.

In [ ]:
calidad = pd.DataFrame(etl.LOG)
descartados = (pd.concat(etl.DESCARTADOS, ignore_index=True) if etl.DESCARTADOS
              else pd.DataFrame(columns=["fuente","asignatura","motivo"]))
print("Registros de calidad:",len(calidad),"| Filas descartadas:",len(descartados))
display(calidad[["fuente","asignatura","problema","registros_afectados","decision"]].head(25))
display(descartados.head(10))

## 11. Validar el modelo

Esta celda ejecuta las comprobaciones que ya implementó Lara: claves únicas, rangos de notas, semáforos, matrícula e integridad de la integración. Una validación `ok=False` requiere análisis antes de avanzar a los KPI.

In [ ]:
personal = pd.concat(personal_participacion + personal_trazabilidad, ignore_index=True)
validaciones = etl.validar_modelo(
    situacion, notas_finales, calificaciones, matricula, universo, personal,
    cursada_cerrada=False
)
display(validaciones)
assert validaciones["ok"].all(), "Existen validaciones fallidas: revisar antes de continuar."

## 12. Ejecutar exactamente el ETL completo (punta a punta)

Hasta acá inspeccionamos cada paso con DataFrames en memoria. Ahora llamamos al orquestador oficial `etl.ejecutar()`, que reproduce el pipeline de producción y escribe las tablas procesadas.

**Nota:** puede generar CSV (y opcionalmente Parquet). Eso no convierte ni modifica las **fuentes originales Excel**.

In [ ]:
SALIDA = Path("/content/etl_resultados")
salidas = etl.ejecutar(str(RAW_DIR), str(SALIDA))
resumen_salidas = pd.DataFrame([{"tabla":nombre, "filas":len(df), "columnas":len(df.columns)}
                                for nombre,df in salidas.items()])
display(resumen_salidas)
print("Archivos generados:",sorted(p.name for p in SALIDA.iterdir()))

## 13. Pruebas automatizadas

Además de la exploración visual, ejecutá los tests existentes con datos sintéticos. Estos tests verifican reglas y casos límite **sin depender de los Excel**.

In [ ]:
resultado = subprocess.run([sys.executable,"-m","pytest","tests/test_etl_6_1.py","-q"],
                            cwd=str(PROJECT_DIR), text=True)
if resultado.returncode:
    raise RuntimeError(f"Fallaron las pruebas (código {resultado.returncode}).")

# Parte II — KPI del punto 6.2

Ya tenemos las tablas limpias del ETL 6.1 en la variable `salidas`. Ahora vamos a **calcular, comprobar y visualizar** los indicadores que pide la consigna.

La lógica de los KPI vive en `src/kpi.py`; el notebook solo llama a las funciones y muestra el resultado. Así evitamos que el código del notebook y los scripts produzcan números diferentes.

**Antes de empezar:** ejecutá las secciones anteriores (en especial la celda `salidas = etl.ejecutar(...)`). Como `src/kpi.py` se agregó recientemente a GitHub, **abrí este notebook desde la rama `matias` y usá un entorno nuevo de Colab** para obtener la última versión del repositorio.

**Criterios importantes:**
- Los KPI estándar usan **matrícula vigente**; los casos sin matrícula aparecen en alertas.
- Los estados son **provisorios, a la fecha del reporte (24/10/2025)**: no implican aprobación final.
- Álgebra tiene nota sobre 120; para comparar entre materias se utiliza porcentaje de la escala.
- `interacciones` (resumen del LMS) y `eventos_trazabilidad` (detalle de eventos) son métricas diferentes. Ninguna equivale automáticamente a sesiones.

## 14. Calcular las 11 tablas KPI

Esta celda invoca el módulo `src.kpi` usando directamente las salidas del ETL **sin releer los Excel**. Para garantizar que el código importado es el correcto, mostramos el archivo desde donde se cargó.

In [ ]:
from src import kpi as modulo_kpi

print("Código KPI cargado desde:", modulo_kpi.__file__)
kpis = modulo_kpi.calcular_kpis(
    salidas["situacion_estudiante"],
    salidas["trazabilidad"],
    salidas["calificaciones"]
)
resumen_kpis = pd.DataFrame([
    {"tabla": nombre, "filas": len(datos), "columnas": len(datos.columns)}
    for nombre, datos in kpis.items()
])
display(resumen_kpis)
assert len(kpis) == 11, "La cantidad de tablas KPI no es la esperada."

## 15. Panorama general: matrícula, actividad y rendimiento

La **cobertura de participación** se calcula como matriculados vigentes con actividad / total matriculados vigentes. El **promedio de nota porcentual** excluye «Sin notas» y NO representa una nota definitiva. El tiempo de uso es una estimación del LMS, no una medición de sesiones.

In [ ]:
resumen = kpis["kpi_resumen_asignatura"].copy()
columnas_resumen = [
    "asignatura", "matriculados", "activos", "inactivos", "cobertura_participacion_pct",
    "con_notas", "promedio_nota_pct_solo_con_notas",
    "tiempo_uso_promedio_min_por_matriculado",
    "interacciones_lms_promedio_por_matriculado"
]
display(resumen[columnas_resumen].round(2))
import plotly.express as px
fig = px.bar(resumen, x="asignatura", y=["activos", "inactivos"],
             barmode="stack", title="Matriculados vigentes: activos vs. inactivos",
             labels={"value": "Estudiantes-materia", "variable": "Estado de actividad"})
fig.show()
assert int(resumen["matriculados"].sum()) == int(salidas["situacion_estudiante"]["matriculado"].sum())

## 16. Estados académicos a la fecha

El semáforo es un **indicador de riesgo académico actual**, no un acta de calificaciones. La categoría «Sin notas» se conserva por separado: no equivale a «Libre». Para calcular porcentajes, el denominador son **todos los matriculados** de esa asignatura.

In [ ]:
estados = kpis["kpi_estados_a_la_fecha"]
display(estados.pivot(index="asignatura", columns="estado_a_la_fecha",
                     values="cantidad").fillna(0).astype(int))
fig = px.bar(estados, x="asignatura", y="porcentaje_sobre_matriculados",
             color="estado_a_la_fecha", barmode="stack",
             title="Situación académica provisional por asignatura",
             labels={"porcentaje_sobre_matriculados": "% de matriculados"})
fig.show()
porcentajes = estados.groupby("asignatura")["porcentaje_sobre_matriculados"].sum()
assert (porcentajes.sub(100).abs() <= 0.05).all(), porcentajes

## 17. Histograma de calificaciones comparables

El ETL usa puntos de 0–120 en Álgebra y 0–100 en las otras dos materias. El KPI normaliza a **0–100 % de su escala** para la visualización comparativa.

**Atención:** este porcentaje incluye las estimaciones que establecieron las reglas de negocio y solo representa el estado **a la fecha**. Los alumnos sin notas no se cuentan como ceros en el histograma.

In [ ]:
hist = kpis["kpi_histograma_notas"]
display(hist.pivot(index="rango_nota_pct", columns="asignatura", values="cantidad").fillna(0))
fig = px.bar(hist, x="rango_nota_pct", y="cantidad", color="asignatura",
             barmode="group", category_orders={
                 "rango_nota_pct": ["0-20", "20-40", "40-60", "60-80", "80-100"]},
             title="Distribución de notas (% de escala), entre quienes tienen nota")
fig.show()
assert int(hist["cantidad"].sum()) == int(resumen["con_notas"].sum())

## 18. Trazabilidad: qué hacen los estudiantes matriculados

Cada fila de trazabilidad es un **evento**, no una sesión. Los eventos potencialmente repetidos se conservan porque los Excel no tienen un identificador único de evento; contamos ambos, tal como define el ETL.

Estas agregaciones de 6.2 se limitan a **matriculados vigentes** para mantener el mismo denominador de los otros KPI.

In [ ]:
tipos = kpis["kpi_interacciones_tipo"]
display(tipos.pivot(index="asignatura", columns="tipo_interaccion",
                   values="eventos").fillna(0).astype(int))
fig = px.bar(tipos, x="asignatura", y="eventos", color="tipo_interaccion",
             barmode="stack", title="Eventos LMS por tipo (matriculados vigentes)")
fig.show()
print("Eventos correspondientes a matriculados:", int(tipos["eventos"].sum()))
print("Eventos de todos los alumnos en ETL:", len(salidas["trazabilidad"]))

## 19. Evolución temporal: día, semana y mes

Las fechas se agrupan según el momento del evento. El conteo de **estudiantes activos** es distinto del conteo de eventos: una misma persona puede generar muchos eventos por día.

Las series incluyen períodos con al menos un evento. Si después necesitamos representar también períodos sin eventos, agregaremos una tabla calendario.

In [ ]:
granularidad = {"Día": ("kpi_actividad_diaria", "dia"),
                "Semana": ("kpi_actividad_semanal", "semana_inicio"),
                "Mes": ("kpi_actividad_mensual", "mes")}
from ipywidgets import interact, Dropdown

def graficar_actividad(periodo="Semana"):
    nombre, eje = granularidad[periodo]
    datos = kpis[nombre].sort_values(["asignatura", eje])
    display(datos.head(8))
    fig = px.line(datos, x=eje, y="eventos", color="asignatura",
                  markers=True, title=f"Evolución de eventos por {periodo.lower()}")
    fig.show()

interact(graficar_actividad, periodo=Dropdown(options=list(granularidad),
                                            value="Semana", description="Período"))

## 20. Mapa de calor de actividad: día y hora

Este gráfico permite identificar **cuándo** ocurren más eventos. El día 0 corresponde al lunes y el día 6 al domingo. Se pueden alternar las asignaturas sin recalcular el ETL.

In [ ]:
from ipywidgets import interact, Dropdown

mapa = kpis["kpi_mapa_calor"]
dias = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
def dibujar_calor(materia):
    d = mapa[mapa["asignatura"] == materia]
    matriz = (d.pivot_table(index="dia_semana", columns="hora", values="eventos",
                            aggfunc="sum", fill_value=0)
              .reindex(index=range(7), columns=range(24), fill_value=0))
    fig = px.imshow(matriz.to_numpy(), x=list(range(24)), y=dias,
                    labels={"x": "Hora", "y": "Día", "color": "Eventos"},
                    aspect="auto", title=f"Mapa de calor — {materia}")
    fig.show()

interact(dibujar_calor, materia=Dropdown(
    options=sorted(mapa["asignatura"].unique()), description="Materia"))

## 21. Rendimiento por instrumento evaluado

Aquí se comparan los instrumentos de cada materia. **Solo se usan las filas marcadas por el ETL como `usa_en_nota_final = True`**, respetando las reglas de recuperatorio confirmadas por el docente.

Los promedios por instrumento son sobre 100 puntos normalizados; cada instrumento puede tener una cantidad distinta de evaluados.

In [ ]:
instrumentos = kpis["kpi_notas_instrumento"]
display(instrumentos.round(2))
fig = px.bar(instrumentos, x="instrumento", y="promedio_nota_100",
             color="asignatura", barmode="group", hover_data=["cantidad_notas"],
             title="Nota media de cada instrumento (0–100)")
fig.update_layout(xaxis_tickangle=-35)
fig.show()

## 22. Cruce analítico: participación y rendimiento

Este gráfico es **descriptivo**. Permite ver si existe una relación entre el tiempo de uso o las interacciones y la nota a la fecha.

El coeficiente de **Pearson** toma valores entre −1 y +1: un valor positivo indica asociación lineal positiva; uno negativo, inversa. No demuestra causalidad. Se calcula solo cuando hay al menos tres pares válidos y variación en ambas variables.

Para facilitar la lectura, la tabla de correlaciones ya viene calculada por asignatura.

In [ ]:
relacion = kpis["kpi_relacion_estudiante"].copy()
display(resumen[["asignatura", "con_notas", "correlacion_tiempo_nota",
                 "correlacion_interacciones_nota", "correlacion_eventos_nota"]])
fig = px.scatter(relacion, x="tiempo_uso_min", y="nota_pct",
                 color="asignatura", hover_data=["id_estudiante"],
                 title="Tiempo estimado de uso vs. nota (%) entre matriculados con nota",
                 labels={"tiempo_uso_min": "Minutos estimados de uso",
                         "nota_pct": "Nota a la fecha (% de escala)"})
fig.show()
print("Pares con nota y tiempo:", int(relacion[["nota_pct", "tiempo_uso_min"]]
                                      .dropna().shape[0]))

## 23. Matrícula y alertas

Son casos que requieren atención institucional, no necesariamente errores de datos.

- **Matriculado sin participación:** tiene inscripción vigente pero no presenta actividad según el ETL.
- **No matriculado con actividad:** aparece en los registros del LMS, aunque no tiene matrícula vigente; puede deberse a situaciones administrativas que requieren revisión.

En el segundo caso sí usamos el **universo completo**, porque precisamente buscamos personas fuera de la matrícula vigente.

In [ ]:
alertas = kpis["kpi_alertas_matricula"]
display(alertas.pivot(index="asignatura", columns="alerta", values="cantidad").fillna(0).astype(int))
fig = px.bar(alertas, x="asignatura", y="cantidad", color="alerta",
             barmode="group", title="Alertas de matrícula por asignatura")
fig.show()
situacion_etl = salidas["situacion_estudiante"]
assert int(alertas.loc[alertas["alerta"] == "No matriculado con actividad",
                      "cantidad"].sum()) == int(
    situacion_etl["situacion_matricula"].eq("NO matriculado con actividad").sum())

## 24. Validar los KPI y ejecutar sus pruebas automatizadas

Antes de dar el 6.2 por cerrado, verificamos que **todas las tablas calculadas respetan los denominadores** y que las pruebas unitarias pasan. Se mostrará la salida completa de `pytest`, incluido el número de pruebas ejecutadas.

No estamos aprobando un PR automáticamente: la revisión y el merge se hacen después.

In [ ]:
from pathlib import Path
import subprocess, sys

assert len(kpis["kpi_relacion_estudiante"]) == int(salidas["situacion_estudiante"]["matriculado"].sum())
assert not kpis["kpi_relacion_estudiante"].duplicated(["id_estudiante", "asignatura"]).any()
assert kpis["kpi_resumen_asignatura"]["matriculados"].sum() == len(kpis["kpi_relacion_estudiante"])
assert int(kpis["kpi_estados_a_la_fecha"]["cantidad"].sum()) == len(kpis["kpi_relacion_estudiante"])
assert int(kpis["kpi_actividad_diaria"]["eventos"].sum()) == int(kpis["kpi_interacciones_tipo"]["eventos"].sum())

resultado_kpi = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_etl_6_1.py", "tests/test_kpi_6_2.py", "-q"],
    cwd=str(PROJECT_DIR), text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print(resultado_kpi.stdout)
assert resultado_kpi.returncode == 0, "Falló pytest: revisar el detalle anterior."
print("Validaciones KPI y pruebas completadas.")

## 25. Guardar los indicadores calculados (opcional)

Podés exportar las 11 tablas KPI a CSV en el entorno temporal de Colab. Estas tablas **no se suben automáticamente a GitHub**, que sigue conservando exclusivamente el código y la documentación.

Para análisis y dashboards conviene usar estas salidas calculadas por el script, no repetir fórmulas en cada gráfico.

In [ ]:
from pathlib import Path

RUTA_KPI = Path("/content/etl_resultados/kpi")
RUTA_KPI.mkdir(parents=True, exist_ok=True)
for nombre, tabla in kpis.items():
    tabla.to_csv(RUTA_KPI / f"{nombre}.csv", index=False, encoding="utf-8-sig")
print(f"Se exportaron {len(kpis)} archivos CSV en: {RUTA_KPI}")
display(pd.DataFrame({"archivo": sorted(p.name for p in RUTA_KPI.glob("*.csv"))}))

## Conclusión: qué tenemos y qué sigue

Al terminar esta Parte II, contamos con **6.1 (ETL) + 6.2 (KPI)** ejecutados en un único notebook. Esto respalda el desarrollo de los siete dashboards del punto 7.

Antes de fusionar los cambios a `main`, revisaremos los números que aparecen en Colab y las pruebas. **No publiques capturas ni resultados con nombres completos de alumnos**; el notebook existente de 6.1 muestra algunos nombres en vistas previas.

**Pendiente de decisión:** la consigna menciona «sesiones promedio». Los datos actuales no incluyen un identificador de sesión, por lo que no se ha creado un número artificial. Si acordamos una regla de inactividad (por ejemplo, 30 minutos sin eventos), podremos introducir una métrica claramente rotulada como *sesiones estimadas* y documentarla.